# ARC-AGI-2 — Kaggle Dry-Run Benchmark (Qwen3-8B 4-bit NF4)

**Objective:** Real hardware measurement of Qwen3-8B 4-bit inference latency, sandbox execution, and 8.5-hour Kaggle budget feasibility.

**Hardware / Accelerator Selection:**
- Select **GPU P100** (16GB VRAM, ~732 GB/s memory bandwidth).
- **Why P100 over T4 x2?** A single P100 provides high memory bandwidth, fits Qwen3-8B 4-bit NF4 comfortably (~5.5GB VRAM used), and avoids bitsandbytes multi-GPU tensor sharding/inter-GPU PCIe latency.
- Internet: **ON** for initial package installation, then run top-to-bottom via **Run All**.

In [ ]:
import os, sys, subprocess, pathlib

# 1. Locate arc-solver repository
CANDIDATES = [
    pathlib.Path('/kaggle/input/arc-solver'),
    pathlib.Path('/kaggle/input/puzzle-solver-arc-solver'),
    pathlib.Path('/kaggle/working/arc-solver'),
    pathlib.Path('.').resolve(),
]
ROOT = None
for p in CANDIDATES:
    if (p / 'src' / 'pipeline.py').exists():
        ROOT = p
        break
assert ROOT is not None, 'arc-solver repo not found — attach as Kaggle dataset or clone to /kaggle/working'
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Using ROOT directory:', ROOT)

# 2. Ensure runtime dependencies are installed
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'bitsandbytes>=0.43.0', 'accelerate>=0.33.0', 'transformers>=4.51.0', 'pyyaml'
])
print('Dependencies verified successfully.')

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device: {device_name} | Total VRAM: {vram_gb:.2f} GB')
    assert vram_gb >= 12.0, f'Warning: {device_name} has only {vram_gb:.1f}GB. P100 16GB is recommended.'
else:
    raise RuntimeError('CUDA is not available. Please enable GPU in the Kaggle notebook settings (Settings -> Accelerator -> GPU P100).')

In [ ]:
from src.llm_client import LLMClient, load_config

cfg_path = ROOT / 'config.kaggle.yaml'
cfg = load_config(cfg_path)
model_name = cfg.get('model_name', 'Qwen/Qwen3-8B')
print(f'Loading {model_name} in 4-bit NF4 using kaggle_local backend...')

client = LLMClient.from_config(cfg)
print('Model and tokenizer successfully loaded into GPU VRAM!')

# Quick smoke generation
smoke_res = client.generate('Reply with exactly: PONG', max_tokens=16, temperature=0.0)
print(f'Smoke test generation: {smoke_res.strip()!r}')

In [ ]:
import random, pathlib
from src.loader import load_puzzle, load_puzzles_from_dir, Puzzle

# Candidate search paths for training puzzles:
# 1. Repo-relative dataset directory
# 2. Kaggle-mounted competition input paths (/kaggle/input/arc-prize-2025/)
data_candidates = [
    ROOT / 'data' / 'arc-agi-2' / 'training',
    ROOT / 'data' / 'training',
    pathlib.Path('/kaggle/input/arc-prize-2025/training'),
    pathlib.Path('/kaggle/input/arc-prize-2025/arc-agi_training_challenges.json'),
    pathlib.Path('/kaggle/input/arc-prize-2024/arc-agi_training_challenges.json'),
]

# Also scan /kaggle/input/ recursively if candidates aren't immediately found
kaggle_input = pathlib.Path('/kaggle/input')
if kaggle_input.exists():
    for p in kaggle_input.glob('**/arc-agi_training_challenges.json'):
        if p not in data_candidates:
            data_candidates.append(p)
    for p in kaggle_input.glob('**/training'):
        if p.is_dir() and p not in data_candidates:
            data_candidates.append(p)

puzzles = []
found_source = None

for pth in data_candidates:
    if not pth.exists():
        continue
    if pth.is_dir():
        all_files = sorted(list(pth.glob('*.json')))
        if all_files:
            random.seed(42)
            sampled_files = random.sample(all_files, min(20, len(all_files)))
            puzzles = [load_puzzle(f) for f in sampled_files]
            found_source = f'Individual JSON directory ({pth}) [Total available: {len(all_files)}]'
            break
    elif pth.is_file() and pth.suffix == '.json':
        all_p = load_puzzles_from_dir(pth)
        if all_p:
            random.seed(42)
            puzzles = random.sample(all_p, min(20, len(all_p)))
            found_source = f'Combined JSON challenge file ({pth}) [Total available: {len(all_p)}]'
            break

assert len(puzzles) == 20, f'Failed to load 20 puzzles. Searched paths: {data_candidates}'
print(f'Data Source Used: {found_source}')
print(f'Loaded {len(puzzles)} benchmark puzzles (seed=42): {[p.id for p in puzzles]}')

In [ ]:
import time
from src.triage import triage_puzzle
from src.brute_force import try_brute_force
from src.pipeline import solve_puzzle, _identity, apply_to_test
from src.sandbox import shutdown_sandbox_pool

results = []
print('=' * 75)
print(f'STARTING {len(puzzles)}-PUZZLE REAL HARDWARE BENCHMARK (Qwen3-8B NF4 on Kaggle GPU)')
print('=' * 75)

for idx, puzzle in enumerate(puzzles, 1):
    t_start = time.time()
    
    # 1. Triage
    triage = triage_puzzle(puzzle)
    
    # 2. Execution path: Brute-Force vs Hopeless-Gate vs LLM Tier 2+
    bf_hit = try_brute_force(puzzle.train_pairs)
    hopeless_gated = False
    needed_llm = False
    guesses = []
    
    if bf_hit is not None:
        mode_str = f'BF_STAGE_{bf_hit.stage}'
        test_out = apply_to_test(bf_hit.code, puzzle.test_inputs[0], timeout_seconds=5.0)
        guesses = [test_out or _identity(puzzle.test_inputs[0]), test_out or _identity(puzzle.test_inputs[0])]
    elif triage.bucket == 'hopeless':
        mode_str = 'HOPELESS_GATE'
        hopeless_gated = True
        guesses = [_identity(puzzle.test_inputs[0]), _identity(puzzle.test_inputs[0])]
    else:
        mode_str = 'LLM_TIER2'
        needed_llm = True
        guesses = solve_puzzle(
            puzzle,
            client,
            time_budget_seconds=90.0,
            n_abstractions=cfg.get('n_abstractions_per_puzzle', 2),
            max_self_debug_retries=cfg.get('max_self_debug_retries', 2),
            n_judges=1,
            early_stop_after_cycles=2,
            sandbox_timeout=5.0,
        )
        
    net_elapsed_s = time.time() - t_start
    
    # Verify correctness against expected test output
    expected = puzzle.test_outputs[0] if puzzle.test_outputs else None
    hit = False
    if expected is not None and guesses:
        hit = any(g == expected for g in guesses)
        
    status_str = 'HIT' if hit else 'MISS'
    print(f'[{idx:02d}/{len(puzzles):02d}] {puzzle.id} | bucket={triage.bucket:<9} | mode={mode_str:<14} | time={net_elapsed_s:5.2f}s | {status_str}')
    
    results.append({
        'puzzle_id': puzzle.id,
        'bucket': triage.bucket,
        'hardness': triage.hardness,
        'mode': mode_str,
        'needed_llm': needed_llm,
        'elapsed_s': net_elapsed_s,
        'hit': hit,
    })

shutdown_sandbox_pool()

In [ ]:
print('\n' + '=' * 75)
print('REAL HARDWARE MEASUREMENT & 8.5-HOUR BUDGET VERIFICATION REPORT')
print('=' * 75)

total_time_s = sum(r['elapsed_s'] for r in results)
avg_time_all_s = total_time_s / len(results)

llm_results = [r for r in results if r['needed_llm']]
avg_time_llm_s = sum(r['elapsed_s'] for r in llm_results) / len(llm_results) if llm_results else 0.0

total_hits = sum(1 for r in results if r['hit'])
hit_rate = (total_hits / len(results)) * 100.0

# 1000-puzzle population extrapolation:
# 2.7% BF (~0.05s) + 13% Hopeless (~0.01s) + 84.3% LLM (avg_time_llm_s)
extrapolated_population_s = (27 * 0.05) + (130 * 0.01) + (843 * avg_time_llm_s)
target_budget_s = 8.5 * 3600.0  # 30,600s
fits = extrapolated_population_s <= target_budget_s
margin_s = target_budget_s - extrapolated_population_s

print(f'Backend Evaluated:                Qwen/Qwen3-8B (4-bit NF4 in-process on GPU)')
print(f'Puzzles Tested:                   {len(results)} (sample seed=42)')
print(f'Total Measured Run Time:          {total_time_s:.2f}s ({total_time_s/60.0:.2f} min)')
print(f'Average Time per Puzzle (All):    {avg_time_all_s:.2f}s')
print(f'Average Time for LLM Puzzles:     {avg_time_llm_s:.2f}s (across {len(llm_results)} LLM-routed puzzles)')
print(f'Measured Sample Hit Rate:         {total_hits}/{len(results)} ({hit_rate:.1f}%)')

print('\n--- 8.5-HOUR (30,600s) KAGGLE BUDGET VERIFICATION ---')
print(f'Target Budget (8.5 Hours):        {target_budget_s:.1f}s (30,600.0s)')
print(f'Extrapolated 1000-Puzzle Time:    {extrapolated_population_s:.1f}s ({extrapolated_population_s/3600.0:.2f} hours)')

if fits:
    print(f'VERDICT (MEASURED ON GPU):        FITS WITH {margin_s/3600.0:.2f} HOURS SAFETY MARGIN ({margin_s/target_budget_s*100.0:.1f}%)')
else:
    print(f'VERDICT (MEASURED ON GPU):        DOES NOT FIT ({abs(margin_s)/3600.0:.2f} HOURS OVER BUDGET)')

# Caveats & Statistical Confidence Checks
print(f'\nLLM-routed puzzle sample size:    {len(llm_results)}')
if len(llm_results) < 15:
    print(f'CAUTION: extrapolation based on only {len(llm_results)} LLM-routed puzzles — treat this as a preliminary signal, not a final verdict. Confirm with a larger sample before relying on this for the actual competition submission strategy.')

print('\nNOTE: The 27/1000 (2.7%) brute-force and 130/1000 (13.0%) hopeless-gate population ratios used in the extrapolation formula are measured from the ARC-AGI-2 TRAINING set (data/arc-agi-2/training) and represent an assumption about the competition evaluation distribution. The actual private test set difficulty distribution is unknown.')